In [ ]:
import requests
import time

# Fő földrajzi típusok Overpass kulcsszavai
geo_types = {
    "beach": "natural=beach",
    "mountain": "natural=peak",
    "lake": "natural=lake",
    "desert": "natural=desert",
    "island": "place=island",
    "attraction": "tourism=attraction",
    "park": "leisure=park", 
    "monument": "historic=monument",
}

def get_geo_scores(city_name, lat, lon, radius=10000):
    """
    Lekérdezi az Overpass API-t, és visszaadja a fő földrajzi típusokra
    a találatok számát normalizált 0-1 skálán.
    """
    scores = {}
    
    for typ, tag in geo_types.items():
        query = f"""
        [out:json][timeout:25];
        (
          node["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          way["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          relation["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
        );
        out center;
        """
        url = "http://overpass-api.de/api/interpreter"
        response = requests.post(url, data={"data": query})
        
        if response.status_code == 200:
            data = response.json()
            count = len(data["elements"])
            scores[typ] = count
        else:
            scores[typ] = -1  # hiba esetén -1
        
        time.sleep(2)  # rate limit elkerülése

    scores = {k: v for k,v in scores.items()}
    
    return scores

# ---------------------------
# Példa városok
cities = [
    {"name": "Barcelona", "lat": 41.3851, "lon": 2.1734},
    {"name": "Lisbon", "lat": 38.7169, "lon": -9.1393},
    {"name": "Tirana", "lat": 41.3275, "lon": 19.8189}
]

for city in cities:
    scores = get_geo_scores(city["name"], city["lat"], city["lon"])
    print(f"\n{city['name']} földrajzi típus pontszámok:")
    for k, v in scores.items():
        print(f"  {k}: {v:.2f}")


In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.numbeo.com/cost-of-living/in/Budapest?displayCurrency=EUR"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Példa: első táblázat cella
table = soup.find("table", {"class": "data_wide_table"})
rows = table.find_all("tr")
for row in rows[:5]:
    cols = row.find_all("td")
    if cols:
        print(cols[0].text.strip(), "->", cols[1].text.strip())


In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome()  # vagy Firefox
driver.get("https://en.climate-data.org/europe/portugal/lisbon/lisbon-3308/")
time.sleep(5)  # várjuk, hogy betöltődjön a táblázat

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
table = soup.find("table", id="small_weather_table")

rows = table.find_all("tr")
for row in rows[:3]:
    cols = row.find_all("td")
    if cols:
        print([c.text.strip() for c in cols])

driver.quit()


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Chrome headless mód
chrome_options = Options()
#chrome_options.add_argument("--headless")

# driver path változtasd meg a saját géped szerint
driver = webdriver.Chrome(options=chrome_options)

url = "https://www.tripadvisor.com/Attractions-g187497-Activities-Barcelona_Catalonia.html"
driver.get(url)

# Várunk, amíg betöltődik az oldal
time.sleep(5)  # egyszerű delay, Selenium WebDriverWait is jobb

# Látnivalók kiválasztása
titles = driver.find_elements(By.CSS_SELECTOR, "div.ZvrsW.N.G")

print("Top 10 látnivaló Barcelonában:")
for t in titles[:10]:
    print("Látnivaló:", t.text.strip())

driver.quit()


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# BUD (47.4369, 19.2556), LIS (38.7742, -9.1342)
dist = haversine(47.4369, 19.2556, 38.7742, -9.1342)
print("BUD-LIS távolság:", round(dist, 1), "km")


In [ ]:
import requests

url = "https://www.unwto.org/tourism-statistics"
response = requests.get(url)

print("UNWTO oldal elérhető:", response.status_code)
# Itt manuálisan kell a letöltött CSV-t feldolgozni, mivel dinamikus.


In [ ]:
# Dummy városadatbázis (0-1 normalizált értékek)

# Attribútumok számításának forrása:
## Földrajz:
## Ár: 
## Klíma:
## Életstílus:
## Távolság:
## Zsúfoltság: 

cities = {
    "Lisbon": {
        "földrajz": {"tengerpart": 0.9, "hegy": 0.2, "város": 0.7, "sziget": 0.5, "tópart": 0.2, "sivatag": 0.1},
        "ár": 0.9,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.4, "relax": 1.0, "aktív": 0.5, "kulturális": 1.0, "családbarát": 0.6},
        "távolság": 1.0,
        "zsúfoltság": 0.5,
    },
    "Barcelona": {
        "földrajz": {"tengerpart": 1.0, "hegy": 0.1, "város": 0.9, "sziget": 0.3, "tópart": 0.2, "sivatag": 0.0},
        "ár": 0.5,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.6, "relax": 0.5, "aktív": 0.3, "kulturális": 1.0, "családbarát": 0.5},
        "távolság": 0.5,
        "zsúfoltság": 1.0,
    },
    "Tirana": {
        "földrajz": {"tengerpart": 0.4, "hegy": 0.7, "város": 0.6, "sziget": 0.2, "tópart": 0.3, "sivatag": 0.0},
        "ár": 0.9,
        "klíma": 0.8,
        "életstílus": {"bulis": 0.3, "relax": 0.5, "aktív": 0.6, "kulturális": 1.0, "családbarát": 0.4},
        "távolság": 1.0,
        "zsúfoltság": 0.2,
    }
}

# Dummy user input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
}

In [ ]:
# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total

In [ ]:
# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")